# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mhassantahir-afk/ML-Engineering-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

rel = "hf://datasets/FlyRank/internship-warehouse"

In [2]:
import numpy as np

feature_frame = con.execute(f"""
    WITH daily AS (
        SELECT
            content_hash_id,
            client_hash_id,
            report_date,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,
            CASE WHEN report_date < DATE '2026-03-16' THEN 'first_half' ELSE 'second_half' END AS period
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
    ),
    features AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(CASE WHEN period = 'first_half' THEN gsc_impressions ELSE 0 END) AS feat_impressions,
            SUM(CASE WHEN period='second_half' THEN gsc_impressions ELSE 0 END) AS feat_impressions_second_half,
            SUM(CASE WHEN period = 'first_half' THEN gsc_clicks ELSE 0 END) AS feat_clicks,
            AVG(CASE WHEN period = 'first_half' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS position_first_half,
            AVG(CASE WHEN period = 'second_half' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS position_second_half,
            SUM(CASE WHEN period = 'first_half' THEN 1 ELSE 0 END) AS feat_days_active,
            SUM(CASE WHEN period = 'first_half' THEN gsc_clicks ELSE 0 END) * 1.0
                / NULLIF(SUM(CASE WHEN period = 'first_half' THEN gsc_impressions ELSE 0 END), 0) AS feat_ctr
        FROM daily
        GROUP BY content_hash_id, client_hash_id
    ),
    label AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(CASE WHEN period = 'first_half' THEN gsc_impressions ELSE 0 END) AS first_half,
            SUM(CASE WHEN period = 'second_half' THEN gsc_impressions ELSE 0 END) AS second_half
        FROM daily
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        f.feat_impressions,
        f.feat_impressions_second_half,
        f.feat_clicks,
        f.position_first_half,
        f.position_second_half,
        (f.position_second_half - f.position_first_half) AS position_change,
        f.feat_days_active,
        f.feat_ctr,
        CASE
            WHEN (l.second_half - l.first_half) * 1.0 / NULLIF(l.first_half, 0) * 100 <= -20
            THEN TRUE ELSE FALSE
        END AS declining_flag
    FROM features f
    JOIN label l
        ON f.content_hash_id = l.content_hash_id AND f.client_hash_id = l.client_hash_id
    WHERE l.first_half > 0
      AND f.position_first_half IS NOT NULL
      AND f.position_second_half IS NOT NULL
    ORDER BY f.content_hash_id, f.client_hash_id
""").df()

# Belt-and-suspenders: also reset the index after sorting, so row positions are fully deterministic
feature_frame = feature_frame.reset_index(drop=True)

print(feature_frame.shape)

print("\n=== Feature Frame ===")
print(feature_frame.shape)
feature_frame.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(139747, 11)

=== Feature Frame ===
(139747, 11)


,content_hash_id,client_hash_id,feat_impressions,feat_impressions_second_half,feat_clicks,position_first_half,position_second_half,position_change,feat_days_active,feat_ctr,declining_flag
0,content_000005d4ced12088,client_9958f0a7ae1df715,23.0,63.0,0.0,72.101852,73.306667,1.204815,9.0,0.000000,False
1,content_00007bd2985b77c3,client_73cda7b4e4f265ea,22.0,25.0,0.0,12.600000,6.600000,-6.000000,12.0,0.000000,False
2,content_0000cd28fbda69f3,client_3ffa76342f366962,11.0,18.0,0.0,4.062500,4.553333,0.490833,8.0,0.000000,False
3,content_00014efc121d911d,client_08a6a72ff48e62c0,53.0,63.0,0.0,6.768864,4.688095,-2.080769,14.0,0.000000,False
4,content_000184dde41afe75,client_62f4a7e64f5e0096,2405.0,2480.0,8.0,3.682934,3.477619,-0.205315,15.0,0.003326,False


In [3]:
feature_frame['high_volume'] = (feature_frame['feat_impressions'] >= 500).astype(int)
feature_frame['position_slipped'] = (feature_frame['position_change'] > 2).astype(int)
feature_frame['score'] = (
    feature_frame['high_volume']
    * feature_frame['position_slipped']
    * feature_frame['feat_impressions']
)

print(feature_frame.shape)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()


(139747, 14)


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: The Anatomy of Growing Content
**Selected Finding:** Growing pages are 37.6% longer (3.2K vs 2.3K words) and 20% younger (184 vs 230 days) than declining pages.

**Where Does the Label Come From:** Trend Direction Up: >10% growth. Down: >10% decline. Stable: within +/-10%. Flat.

* **Methodology Question(Label & Window Alignment):** Were the `count`, `words` and `age` captured as single static values at the end of the 30-day impression Window or were they capture prior to the window? If it were to be captured at the end of the impression window at the time of evaluation it is natural for the upwards trending pages to show higher word count and lower age, it would be unclear whether those were the driving factors in the upwards trend or the result of the update.
* **Methodology Question(Validation Design):** what was the split Design for the validation of this claim? Was this word-count pattern tested across grouped client splits, or calculated as a simple portfolio-wide average? If a large enterprise client was present which naturally produces larger word count articles and hold higher baseline domain authority, then this un-grouped global split conflates overall client strength with the effect of page length.

### Finding 9: Captured Traffic Value
**Selected Finding:** Transactional intent leads captured click-equivalent value (\$92.0K), followed by commercial intent (\$74.9K), demonstrating higher value density than raw traffic volume.

**Where Does the Label Come From:** The Target metric `captured click-equivalent value` comes from clicks × CPC(cost-per-click).

* **Methodology Question(Label Origin):** Where does the label CPC originate from? The origin of the Label CPC is not clear in the paper. Mathematically speaking, If transactional keywords naturally carry a \$0.76 CPC baseline while informational keywords carry \$0.15, multiplying clicks by CPC guarantees transactional intent will lead in total value density by mathematical design.

* **Methodology Question(Validation Design):** What was the Split Design for the validation of this claim? There isn't any information on the client distributions between these groups. Was this value distribution validated across all clients equally? it is entirely possible that the transactional group revenue is entirely dominated by a few small number of High Volume e-commerece giants.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

# Reuse the feature_frame already built earlier in this notebook
feature_cols = ['feat_impressions', 'feat_clicks', 'position_first_half',
                'feat_days_active', 'feat_ctr']

X = feature_frame[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = feature_frame['declining_flag']
groups = feature_frame['client_hash_id']  # used ONLY for splitting, never as a feature

splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]


#Random Split (matching test set size from grouped split)
#test_ratio_random = len(X_test) / len(X)  # matches ~8.3% test size
'''train_idx_random, test_idx_random = train_test_split(
    np.arange(len(X)),
    test_size=test_ratio_random,
    random_state=42,
    stratify=y
)'''

train_idx_random, test_idx_random = train_test_split(
    np.arange(len(X)),
    test_size=0.3,
    random_state=42
)

X_train_random = X.iloc[train_idx_random]
X_test_random = X.iloc[test_idx_random]
y_train_random = y.iloc[train_idx_random]
y_test_random = y.iloc[test_idx_random]

print(f"Train size (random): {len(X_train_random)} ({len(X_train_random)/len(X)*100:.1f}%)")
print(f"Test size (random): {len(X_test_random)} ({len(X_test_random)/len(X)*100:.1f}%)")
print(f"Train decline rate (random): {y_train_random.mean():.3f}")
print(f"Test decline rate (random): {y_test_random.mean():.3f}")

Train size (random): 97822 (70.0%)
Test size (random): 41925 (30.0%)
Train decline rate (random): 0.277
Test decline rate (random): 0.282


In [11]:
# 3. Decision Tree Model (Random Split)
tree_random = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree_random.fit(X_train_random, y_train_random)
tree_scores_test_random = tree_random.predict_proba(X_test_random)[:, 1]

# 4. Random Forest Model (Random Split)
rf_random = RandomForestClassifier(n_estimators=200, max_depth=4, class_weight="balanced", random_state=42)
rf_random.fit(X_train_random, y_train_random)
rf_scores_test_random = rf_random.predict_proba(X_test_random)[:, 1]

# 5. Baseline Scores on Random Test Set
baseline_scores_test_random = feature_frame.loc[X_test_random.index, 'score'].values
base_rate_test_random = y_test_random.mean()

# 6. Compute Precision@K Comparison Table for Random Split
k_values_random = [20, 50]

methods_scores_random = {
    'Baseline rule': baseline_scores_test_random,
    'Decision Tree': tree_scores_test_random,
    'Random Forest': rf_scores_test_random,
}

rows_random = []
for method, scores in methods_scores_random.items():
    row = {'method': method}
    for k in k_values_random:
        row[f'precision_at_{k}'] = precision_at_k(scores, y_test_random.values, k)
    rows_random.append(row)

base_row_random = {'method': 'Base rate'}
for k in k_values_random:
    base_row_random[f'precision_at_{k}'] = base_rate_test_random
rows_random.append(base_row_random)

comparison_table_random = pd.DataFrame(rows_random)
print("\n--- Random Split Results ---")
print(comparison_table_random)


--- Random Split Results ---
          method  precision_at_20  precision_at_50
0  Baseline rule         0.400000         0.320000
1  Decision Tree         0.450000         0.360000
2  Random Forest         0.950000         0.820000
3      Base rate         0.281694         0.281694


In [7]:
#Honest Split (Client Based)

#Baseline Score on Honest Split
baseline_scores_test = feature_frame.loc[X_test.index, 'score'].values
baseline_precision_50_test = precision_at_k(baseline_scores_test, y_test.values, 50)

base_rate_test = y_test.mean()

#Decision Tree on Honest Split
tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree.fit(X_train, y_train)

tree_scores_test = tree.predict_proba(X_test)[:, 1]
tree_precision_50 = precision_at_k(tree_scores_test, y_test.values, 50)

rf = RandomForestClassifier(n_estimators=200, max_depth=4, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)

rf_scores_test = rf.predict_proba(X_test)[:, 1]
rf_precision_50 = precision_at_k(rf_scores_test, y_test.values, 50)

# Precision at multiple k values for all methods
k_values = [20, 50]

methods_scores = {
    'Baseline rule': baseline_scores_test,
    'Decision Tree': tree_scores_test,
    'Random Forest': rf_scores_test,
}

rows = []
for method, scores in methods_scores.items():
    row = {'method': method}
    for k in k_values:
        row[f'precision_at_{k}'] = precision_at_k(scores, y_test.values, k)
    rows.append(row)

# Base rate is k-independent, but shown for reference at each k
base_row = {'method': 'Base rate'}
for k in k_values:
    base_row[f'precision_at_{k}'] = base_rate_test
rows.append(base_row)

comparison_table = pd.DataFrame(rows)
print(comparison_table)

          method  precision_at_20  precision_at_50
0  Baseline rule         0.250000         0.240000
1  Decision Tree         0.400000         0.520000
2  Random Forest         0.500000         0.460000
3      Base rate         0.301134         0.301134


I re-ran my week-5 model on a random split and my original client based split.

the difference in baseline scores(0,25 to 0.40 in Precision@20) and Random forest(0.50 to 0.95 in Precision@20) was well expected as now the model can learn client specific characteristics and cheat through the test phase thus granting it a higher score.

although the Decision Tree model seems to be doing worse in the random split, my guess is because of my max_depth cap at 4 which voids it of high complexity thus random split decision tree shows worse results.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [12]:
leaky_feature_set = ['feat_impressions', 'feat_clicks', 'position_first_half',
                     'feat_days_active', 'feat_ctr', 'feat_impressions_second_half']

X_leaky = feature_frame[leaky_feature_set].replace([np.inf, -np.inf], np.nan).fillna(0)
y_leaky = feature_frame['declining_flag']
groups = feature_frame['client_hash_id']

splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
leaky_train_idx, leaky_test_idx = next(splitter.split(X_leaky, y_leaky, groups=groups))

# FIX: use leaky_train_idx / leaky_test_idx consistently everywhere below
X_train_leaky, X_test_leaky = X_leaky.iloc[leaky_train_idx], X_leaky.iloc[leaky_test_idx]
y_train_leaky, y_test_leaky = y_leaky.iloc[leaky_train_idx], y_leaky.iloc[leaky_test_idx]

baseline_scores_test_leaky = feature_frame.loc[X_test_leaky.index, 'score'].values
baseline_precision_50_leaky = precision_at_k(baseline_scores_test_leaky, y_test_leaky.values, 50)
base_rate_test_leaky = y_test_leaky.mean()

tree_leaky = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree_leaky.fit(X_train_leaky, y_train_leaky)
tree_scores_test_leaky = tree_leaky.predict_proba(X_test_leaky)[:, 1]

rf_leaky = RandomForestClassifier(n_estimators=200, max_depth=4, class_weight="balanced", random_state=42)
rf_leaky.fit(X_train_leaky, y_train_leaky)
rf_scores_test_leaky = rf_leaky.predict_proba(X_test_leaky)[:, 1]

k_values = [20, 50]
methods_scores_leaky = {
    'Baseline rule': baseline_scores_test_leaky,
    'Decision Tree (with leak)': tree_scores_test_leaky,
    'Random Forest (with leak)': rf_scores_test_leaky,
}

rows = []
for method, scores in methods_scores_leaky.items():
    row = {'method': method}
    for k in k_values:
        row[f'precision_at_{k}'] = precision_at_k(scores, y_test_leaky.values, k)
    rows.append(row)

base_row = {'method': 'Base rate'}
for k in k_values:
    base_row[f'precision_at_{k}'] = base_rate_test_leaky
rows.append(base_row)

comparison_table_leaky = pd.DataFrame(rows)
print(comparison_table_leaky)

                      method  precision_at_20  precision_at_50
0              Baseline rule         0.250000         0.240000
1  Decision Tree (with leak)         1.000000         1.000000
2  Random Forest (with leak)         1.000000         1.000000
3                  Base rate         0.301134         0.301134


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.